In [1]:
#kc-event-explorer/
#│
#├── WIP- 1.A.py                  
#├── templates/
#│   └── index.html
#│
#├── static/
#│   ├── style.css
#│   └── script.js
#│
#├── data/
#│   └── neighborhoods.geojson
#│
#├── services/
#│   ├── ticketmaster.py
#│   └── geocoder.py
#│
#└── database/
#    └── events.db

In [2]:
#struggles: I tried to add the filter by city

In [3]:
#import flask, import API keys from keys.json, import requests
import requests
from flask import Flask, render_template, request
from keys import TICKETMASTER_KEY, EVENTBRITE_TOKEN

app = Flask(__name__)



In [4]:
def get_city(event):
    return (
        event.get("city")
        or event.get("_embedded", {})
            .get("venues", [{}])[0]
            .get("city", {})
            .get("name")
    )

In [ ]:
#pull events from Ticketmaster 
def get_ticketmaster_events(city):
    params = {
        "apikey": TICKETMASTER_KEY,
        "size": 50
    }

    if city:
        params["city"] = city

    response = requests.get(
        "https://app.ticketmaster.com/discovery/v2/events.json",
        params=params
    )

    data = response.json()
    return data.get("_embedded", {}).get("events", [])


In [6]:
#pull events from Eventbrite
def get_eventbrite_events():
    url = "https://www.eventbriteapi.com/v3/events/search/"

    headers = {
        "Authorization": f"Bearer {EVENTBRITE_TOKEN}"
    }

    params = {
        "location.address": request.args.get("city", ""),
        "page_size": 50
    }

    response = requests.get(url, headers=headers, params=params)
    return response.json()

In [7]:
#normalize Ticketmaster data
def normalize_ticketmaster(data):
    events = []

    if "_embedded" in data:
        for event in data["_embedded"]["events"]:

            start = event.get("dates", {}).get("start", {})

            raw_time = start.get("localTime")
            time = raw_time[:5] if raw_time else None

            venue = {}

            if "_embedded" in event:
                venues = event["_embedded"].get("venues", [])
                if venues:
                    venue = venues[0]

            events.append({
                "name": event.get("name"),
                "date": start.get("localDate"),
                "time": time,
                "source": "ticketmaster",
                "lat": venue.get("location", {}).get("latitude"),
                "lon": venue.get("location", {}).get("longitude"),
                "city": venue.get("city", {}).get("name") 
            })

    return events

In [8]:
#normalize Eventbrite data
def normalize_eventbrite(data):
    events = []

    for event in data.get("events", []):

        name = event.get("name", {}).get("text")
        start = event.get("start", {}).get("local")

        date = None
        time = None
        city = event.get("venue", {}).get("address", {}).get("city")

        if start:
            date = start[:10]
            time = start[11:16]

        events.append({
            "name": name,
            "date": date,
            "time": time,
            "source": "eventbrite",
            "lat": None,
            "lon": None,
            "city": city 
        })

    return events

from datetime import datetime


In [9]:
#get all events from both sources and normalize 
def get_all_events():
    tm_data = get_ticketmaster_events()
    eb_data = get_eventbrite_events()

    events = []

    events += normalize_ticketmaster(tm_data)
    events += normalize_eventbrite(eb_data)

    return events


In [10]:
#home route to display events on webpage
@app.route("/")
def home():

    print("ROUTE HIT")

    selected_date = request.args.get("date")
    selected_time = request.args.get("time")
    selected_city = request.args.get("city")

    data = get_all_events()

    # dropdown values 
    cities = sorted({
        e.get("city")
        for e in data
        if e.get("city")
    })

    # filtering 
    filtered = data

    if selected_city:
        filtered = [e for e in filtered if e.get("city") == selected_city]

    if selected_date:
        filtered = [e for e in filtered if e.get("date") == selected_date]

    if selected_time:
        filtered = [
            e for e in filtered
            if e.get("time") and e.get("time") >= selected_time
        ]

    # build final display list
    events = []

    for event in filtered:

        name = event.get("name")
        date = event.get("date")
        time = event.get("time")
        city = event.get("city")

        display_time = "TBD"

        if time:
            try:
                display_time = datetime.strptime(time, "%H:%M").strftime("%I:%M %p")
            except:
                display_time = time

        events.append({
            "name": name,
            "date": date,
            "time": display_time,
            "city": city,
            "source": event.get("source"),
            "lat": event.get("lat"),
            "lon": event.get("lon")
        })

    print("EVENT COUNT:", len(events))

    return render_template(
        "index.html",
        events=events,
        cities=cities,
        selected_city=selected_city,
        selected_date=selected_date,
        selected_time=selected_time
    )

In [11]:
#run Flask app
if __name__ == "__main__":
    app.run(host="127.0.0.1", port=5001, debug=True, use_reloader=False)


 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5001
Press CTRL+C to quit


ROUTE HIT


127.0.0.1 - - [01/Jul/2026 07:55:06] "GET / HTTP/1.1" 200 -


EVENT COUNT: 100
